In [ ]:
from cosipy.nonimaging.bgo.bc_tools_localization import BGOLocalizerBCT
import numpy as np


In [ ]:
# Inizializza con i tre LUT (pickle) e nside

dir_path = "/data/test_newrepo/"
run_name="run10"
localizer = BGOLocalizerBCT(
    soft_loctable_path=dir_path+'/soft_local_loc_table_' + run_name + '.pkl',
    medium_loctable_path=dir_path+'/medium_local_loc_table_' + run_name + '.pkl',
    hard_loctable_path= dir_path+'/hard_local_loc_table_' + run_name + '.pkl',
    nside=64,
)
    

In [ ]:
def ra_dec_to_theta_phi(ra, dec):

    theta = 90-dec
    
    phi = ra
    
    return theta, phi
def spherical_to_radec_deg(theta_deg, phi_deg):

    dec = 90.0 - theta_deg
    ra = phi_deg % 360.0
    return ra, dec


In [ ]:
# Counts order ['BGO_Z1','BGO_Z0','BGO_X1','BGO_X0','BGO_Y1','BGO_Y0']
# Simulated GRB
# Spectrum Band 10 10000 -1.9 -3.7 230
# Flux 14.58 ph/cm2/s
# True position theta=84.021 phi=49.922

true_ra,true_dec = spherical_to_radec_deg(84.021,49.922)
# s_counts = [46, 316, 33, 374, 47, 34]

# #for testing purpose generate random Poissonian background 
# mean_counts = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
# random_bkg = np.random.poisson(np.array([mean_counts[3],mean_counts[2],mean_counts[5],mean_counts[4],mean_counts[1],mean_counts[0]])*20)

# b_counts = np.array([mean_counts[3],mean_counts[2],mean_counts[5],mean_counts[4],mean_counts[1],mean_counts[0]])*20
# s_counts = s_counts+random_bkg


#l,b = 0,0
#s_counts = np.array([1593., 1637., 2737., 2546., 2884., 3015.])
#b_counts = np.array([1089.83682979, 1098.7535547 , 1131.83335108, 1079.34888392,1323.91390867, 1277.07061723])

#l,b =  9.622 -30.350
#s_counts = np.array([1780., 1665., 2524., 2546., 2688., 2743.])
#b_counts = np.array([1089.83682979, 1098.7535547 , 1131.83335108, 1079.34888392,1323.91390867, 1277.07061723])

#l,b = 90 0
#s_counts = np.array([1583., 1278., 1877., 1689., 2307., 4893.])
#b_counts = np.array([1049.16566569, 1060.2962449 , 1088.80749725, 1039.6356575 ,1275.06287692, 1227.71327162])

#bn210528586.fits
s_counts = np.array([2979, 3154, 3663, 3784, 4365, 3444])
b_counts = np.array([821.30780687,  822.60080117,  890.55696027,  938.97155898, 1038.89648148, 1038.82905046])


# z1: signal=1951.00, background=1089.84, net=861.16
# z0: signal=1924.00, background=1098.75, net=825.25
# x1: signal=2431.00, background=1131.83, net=1299.17
# x0: signal=2356.00, background=1079.35, net=1276.65
# y1: signal=2863.00, background=1323.91, net=1539.09
# y0: signal=3116.00, background=1277.07, net=1838.93

In [ ]:

from astropy.coordinates import SkyCoord
import astropy.units as u
from scoords import Attitude, SpacecraftFrame
ori_file  = "/home/cosi/cosi/data/background/dc4/DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"

data = np.loadtxt(ori_file, usecols=(1, 2, 3, 4, 5, 6, 7, 8), delimiter=' ', skiprows=1, comments=("#", "EN"))
start_ori = 1835487300.0

In [ ]:

time_grb = 2096.75 +start_ori

tempi = data[:, 0]

idx = np.argmin(np.abs(tempi - time_grb))

nearest_row = data[idx]

print("Index:", idx)
print("Nearest ori bin:", tempi[idx])
print("Row", nearest_row)

i = idx

In [ ]:

print(data[:, 2][i]*u.deg) 
print(data[:, 1][i]*u.deg)

print(data[:, 4][i]*u.deg) 
print(data[:, 3][i]*u.deg)


x_pointing = SkyCoord(data[:, 2][i]*u.deg, data[:, 1][i]*u.deg, frame='galactic')
z_pointing = SkyCoord(data[:, 4][i]*u.deg, data[:, 3][i]*u.deg, frame='galactic')
#x_pointing = SkyCoord(0*u.deg, 0*u.deg, frame='galactic')
#z_pointing = SkyCoord(0*u.deg, 90*u.deg, frame='galactic')
attitude = Attitude.from_axes(x=x_pointing, z=z_pointing, frame='icrs')

result = localizer.localize(s_counts, b_counts,attitude=attitude,duration=1.28)
print(result)

In [ ]:
result

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

grb_l = 219.027
grb_b = -38.527

# due coordinate galattiche (l, b)
c1 = SkyCoord(l=result['l']*u.deg, b=result['b']*u.deg, frame='galactic')
c2 = SkyCoord(l=grb_l*u.deg, b=grb_b*u.deg, frame='galactic')

# distanza angolare
sep = c1.separation(c2)

print(sep.deg)

In [ ]:
true_coord = SkyCoord(l=result['l']*u.deg, b=result['b']*u.deg, frame='galactic')

ts_map = result["ts_map"]
img, ax = ts_map.plot()
ax.grid(alpha=0.5)

""" 
if true_coord is not None:
    # Actual location of simulated source
    ax.scatter(
        true_coord.icrs.ra.to(u.deg).value,
        true_coord.icrs.dec.to(u.deg).value,
        color="red",
        transform=ax.get_transform("world"),
        s=2,
        label="True source"
    )
best_loc = SkyCoord(l=result['l']*u.deg, b=result['b']*u.deg, frame="galactic")
ax.scatter(
        
        best_loc.icrs.ra.to(u.deg).value,
        best_loc.icrs.dec.to(u.deg).value,
        color="blue",
        transform=ax.get_transform("world"),
        s=2,
        label="Best localization"
    )
 """
# Add legend 
ax.legend(loc="upper right", frameon=True)

In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

nside = 32
npix = hp.nside2npix(nside)


m = ts_map.data

hp.projview(
    m,
    coord=["G"],
    graticule=True,
    graticule_labels=True,
    unit="deg2",
    xlabel="longitude",
    ylabel="latitude",
    cb_orientation="vertical",
    latitude_grid_spacing=30,
    projection_type="aitoff",
    title="Aitoff projection",
    cmap="turbo",
    nest=False
)

plt.show()
